# 02. Chia dữ liệu, so sánh cấu hình và train

Mọi lựa chọn (bộ phân loại, align, augment, ngưỡng) đều dựa trên tập validation. Notebook này không đụng tới tập test, phần chấm test để ở notebook 03 và chỉ chạy một lần.

Các bước gọi đúng các hàm của `ml.prepare_data`, `ml.compare`, `ml.train_baseline`, `ml.learning_curve`, nên kết quả giống khi chạy lệnh trong terminal.

In [ ]:
# Mở notebook từ thư mục notebooks/ hay thư mục gốc đều chạy được
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Audio, Image, Markdown, display
pd.set_option('display.max_colwidth', 80)
print('Thư mục làm việc:', ROOT)

In [ ]:
# Tham số, sửa lại cho đúng dữ liệu
def _env_list(name, default):
    value = os.environ.get(name)
    return value.split(',') if value else default

MANIFEST   = os.environ.get('VOVC_MANIFEST', 'data/manifest.csv')
SPLIT      = os.environ.get('VOVC_SPLIT', 'data/manifest_split.csv')
VALIDATION = _env_list('VOVC_VALIDATION', ['p01/s02'])   # 1 người: phiên riêng; 2 người: ['p01/s02']
TEST       = _env_list('VOVC_TEST', ['p01/s03'])         # 1 người: phiên riêng; 2 người: ['p02']
MODEL_OUT  = os.environ.get('VOVC_MODEL', 'models/nb_model_01.joblib')
AUGMENTS   = [0, 3]          # các mức augment đem so sánh
CHOICE     = None            # None = lấy cấu hình tốt nhất; hoặc dict(classifier='logreg', align=False, augment=0)

## 1. Chia train / validation / test

In [ ]:
from app.common import LABELS
from ml.dataset import read_manifest
from ml.prepare_data import split_manifest

if Path(SPLIT).exists():
    print(f'Dùng lại {SPLIT} (xóa file này nếu muốn chia lại).')
else:
    sessions = {f"{r['speaker_id']}/{r['session_id']}" for r in read_manifest(MANIFEST, require_splits=False)}
    speakers = {s.split('/')[0] for s in sessions}
    need = [u for u in VALIDATION + TEST if u not in sessions and u not in speakers]
    if need:
        raise RuntimeError(f'Chưa có {need}. Hiện có: {sorted(sessions)}. Thu thêm phiên '
                           f'(xem README) hoặc sửa VALIDATION/TEST ở ô tham số rồi chạy lại.')
    print(split_manifest(MANIFEST, SPLIT, validation=VALIDATION, test=TEST))

split_rows = read_manifest(SPLIT)
split_df = pd.DataFrame([{k: r[k] for k in ['speaker_id', 'session_id', 'label', 'split']} for r in split_rows])
pd.crosstab(split_df.split, split_df.label).reindex(index=['train', 'validation', 'test'], columns=LABELS, fill_value=0)

## 2. So sánh cấu hình trên validation

Thử 3 bộ phân loại, có và không có align, với các mức augment. Validation ít mẫu thì chênh vài điểm có thể chỉ là ngẫu nhiên, nếu ngang nhau thì chọn cấu hình đơn giản hơn (logreg, không augment).

In [ ]:
from ml.compare import compare
from ml import viz

results = compare(SPLIT, augments=AUGMENTS)
viz.compare_bars(results)
plt.show()
pd.DataFrame(results)

## 3. Train cấu hình đã chọn

In [ ]:
import json
from ml.train_baseline import train

chosen = CHOICE or {k: results[0][k] for k in ['classifier', 'align', 'augment']}
print('Cấu hình:', chosen)
training_json = Path(MODEL_OUT).with_suffix('.training.json')
if Path(MODEL_OUT).exists():
    print(f'{MODEL_OUT} đã có, nạp lại báo cáo cũ (đổi MODEL_OUT nếu muốn train mới).')
    report = json.loads(training_json.read_text(encoding='utf-8'))
else:
    report = train(SPLIT, MODEL_OUT, **chosen)
print(f"Ngưỡng {report['selected']['threshold']:.3f}, macro-F1 validation {report['selected']['macro_f1']:.3f}")
viz.threshold_curve(report)
plt.show()

## 4. Learning curve: thu thêm có đáng không

Train lại với 20% tới 100% dữ liệu train (mỗi mức lặp 3 lần với tập con ngẫu nhiên), chấm trên validation. Nếu bên phải đường vẫn còn đi lên thì thu thêm dữ liệu vẫn có ích. Nếu đã đi ngang thì nên đổi cách khác (thêm người, thêm điều kiện, đổi đặc trưng) hơn là thu thêm y như cũ.

In [ ]:
from ml.learning_curve import curve

points = curve(SPLIT, [0.2, 0.4, 0.6, 0.8, 1.0], repeats=3, **chosen)
viz.learning_curve(points)
plt.show()
pd.DataFrame(points)

## 5. Xem model đoán sai gì trên validation

In [ ]:
from ml.predict import Predictor

predictor = Predictor(MODEL_OUT)
val_rows = [r for r in split_rows if r['split'] == 'validation']
pred = pd.DataFrame([{'path': r['path'], 'expected': r['label'], **predictor.predict_file(r['_file']).to_dict(),
                      'file': r['_file']} for r in val_rows])
wrong = pred[pred.label != pred.expected]
print(f'Validation: đúng {len(pred) - len(wrong)}/{len(pred)}')
pd.crosstab(pred.expected, pred.label).reindex(index=LABELS, columns=LABELS, fill_value=0)

In [ ]:
for _, r in wrong.head(6).iterrows():
    display(Markdown(f"`{r.path}`: thật **{r.expected}**, đoán **{r.label}** (gần nhất {r.raw_label}, {r.score:.2f})"))
    display(Audio(filename=r.file))

## Làm tiếp

- Ổn rồi thì sang notebook 03 để chấm test (chỉ một lần).
- Muốn dùng model này trong GUI: `python -m app.gui --model models/nb_model_01.joblib` (đổi tên theo `MODEL_OUT`).
- Trước khi commit nhớ Clear All Outputs.